# Phase 1c: A Driving Scene from WayveScenes101

One 20-second drive recorded by **5 synchronised fisheye cameras** (1,000 images), with poses and masks supplied by Wayve.

**What's new compared with the garden and phone video**
| Challenge | How we handle it |
|---|---|
| Fisheye lenses | Undistort with OpenCV's fisheye model, which matches the `OPENCV_FISHEYE` cameras in the files. gsplat's built-in fisheye handling uses a simplified model that's off by ~80 px on this data (`docs/failures.md` F1.6) |
| Blurred faces and plates, the car's own body | Wayve's masks, plus the black edges left by undistortion, are excluded from training and scoring (small patch to gsplat's data loader) |
| A fair test | Wayve's own protocol: train on the 4 side cameras, test on the **front-forward camera**, which is never trained on. It's a harder, *off-axis* test than garden's |
| Moving objects | We picked `scene_078`, which Wayve's metadata tags as having no traffic, pedestrians or cyclists, and no large exposure change |

**No COLMAP here:** the poses come with the dataset. We still check them by reprojecting Wayve's 3D points.

⏱️ Rough total on a T4: 35-60 min (download ~1-3 min, preparation ~3 min, training 15k steps ~20-35 min). The first session after a Colab update also spends 10-20 min compiling gsplat once.

## ⚖️ License: read this first

The data is under the **WayveScenes101 Dataset Licence Agreement for Non-Commercial Use (June 2024)**. Downloading it means agreeing to it, so read the full text first: it's in the same Google Drive folder as the scenes, and linked from https://wayve.ai/science/wayvescenes101/.

Points that matter for this project (a summary, not legal advice):
- **Non-commercial only:** personal experimentation, research and teaching. Nothing aimed at commercial advantage.
- **Attribution is required** in anything made with it, word for word: *"This work was made using the WayveScenes101 dataset, provided by Wayve Technologies Ltd under licence terms available at https://wayve.ai/science/wayvescenes101/"*
- **Don't publish the trained models** (the `.ply` files) unless it's on an academic or scientific basis and free of charge. Keep them private and show **videos or screenshots** instead.
- **Not for operating vehicles or robots**, or for high-risk uses. Keep the Phase 4 navigation agent on your own scenes, not Wayve's.

## 1. GPU, Drive, settings

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
from google.colab import drive
drive.mount('/content/drive')

import os, sys, glob, json, time, re, subprocess, shutil
SCENE       = 'scene_078'   # tagged: overcast, residential, no traffic/pedestrians, no large exposure change
FACTOR      = 2             # 1920x1080 -> 960x540
ZOOM        = 0.8           # side cameras: keep more of the wide view (front camera stays at 1.0)
STEPS       = 30_000        # gsplat's standard schedule ...
SCALE       = 0.5           # ... run at half length (15k steps) with every phase scaled to match
TEST_STRIDE = 2             # score every 2nd front-forward frame (100 of 200)

PROJECT = '/content/drive/MyDrive/3dgs-digital-twin'
WORK    = '/content/work'
OUT     = f'{PROJECT}/results/wayve_{SCENE}'
CACHE   = f'{PROJECT}/datasets/wayve'    # private copy of the scene zip, so we don't re-download it
for d in [f'{PROJECT}/results', f'{PROJECT}/exports', CACHE, WORK, OUT]:
    os.makedirs(d, exist_ok=True)

## 2. Install gsplat

The same install cell as the other notebooks, with every fix from `docs/failures.md` built in. COLMAP isn't needed for this scene.

In [ ]:
import importlib.util
MARKER = '/content/.gs_deps_ok'          # /content survives a kernel restart (not a VM reset)

if not os.path.exists(MARKER):
    import torch
    np_before = subprocess.run([sys.executable, '-c', 'import numpy; print(numpy.__version__)'],
                               capture_output=True, text=True).stdout.strip()
    # gsplat: prebuilt wheel if available, else PyPI (JIT compile)
    tv = torch.__version__.split('+')[0].split('.'); cu = (torch.version.cuda or '').replace('.', '')
    r = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'gsplat==1.5.3',
                        '--index-url', f'https://docs.gsplat.studio/whl/pt{tv[0]}{tv[1]}cu{cu}',
                        '--extra-index-url', 'https://pypi.org/simple'], capture_output=True, text=True)
    if r.returncode != 0:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'gsplat==1.5.3'], check=True)

    REPO = f'{WORK}/gsplat'
    if not os.path.exists(REPO):
        subprocess.run(['git', 'clone', '-q', '--depth', '1', '--branch', 'v1.5.3',
                        'https://github.com/nerfstudio-project/gsplat.git', REPO], check=True)
    reqs = open(f'{REPO}/examples/requirements.txt').read()

    # F0.1 + F0.2: git deps (several glued onto one line) -> split, build without isolation
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'ninja', 'setuptools', 'wheel'], check=True)
    GPU_NAME = torch.cuda.get_device_name(0)
    if 'T4' in GPU_NAME:
        os.environ['TORCH_CUDA_ARCH_LIST'] = '7.5'   # faster builds, T4 only
    for url in re.findall(r'git\+https://github\.com/[^@\s]+@[0-9a-f]{40}', reqs):
        print('building', url.split('/')[-1][:60])
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-build-isolation', url], check=True)

    # F0.3: drop git lines + numpy<2 pin, install the rest
    rest = [l for l in reqs.splitlines() if 'git+https' not in l and not l.lower().startswith('numpy')]
    open('/tmp/reqs.txt', 'w').write('\n'.join(rest))
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', '/tmp/reqs.txt'], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'remotezip', 'plyfile', 'numpy>=2.1,<2.5'], check=True)

    # F0.5: pycolmap fork uses np.uint64(-1) -> OverflowError on numpy 2 (patch the file, don't import it)
    f = os.path.join(os.path.dirname(importlib.util.find_spec('pycolmap').origin), 'scene_manager.py')
    s = open(f).read()
    if 'np.uint64(-1)' in s:
        open(f, 'w').write(s.replace('np.uint64(-1)', 'np.uint64(18446744073709551615)')); print('patched pycolmap')

    # F1.5: otherwise gsplat recompiles its CUDA code at the first render of every session (~13 min).
    # Build a compiled package once, keep it on Drive, and install that in later sessions.
    if not subprocess.run([sys.executable, '-c', 'from gsplat import csrc'], capture_output=True).returncode == 0:
        WHEELS = (f"{PROJECT}/cache/wheels/gsplat153_pt{torch.__version__.split('+')[0]}_cu{cu}"
                  f"_py{sys.version_info.major}{sys.version_info.minor}_{GPU_NAME.replace(' ', '-')}")
        os.makedirs(WHEELS, exist_ok=True)
        if not glob.glob(f'{WHEELS}/gsplat-*.whl'):
            print('Compiling gsplat once for this GPU + torch (10-20 min). Later sessions reuse it.')
            r = subprocess.run([sys.executable, '-m', 'pip', 'wheel', '-q', '--no-deps', '--no-build-isolation', '-w', WHEELS,
                                'git+https://github.com/nerfstudio-project/gsplat.git@v1.5.3'],
                               capture_output=True, text=True)
            if r.returncode != 0:
                print('Building failed, gsplat will compile during training instead:', r.stderr[-800:])
        for whl in glob.glob(f'{WHEELS}/gsplat-*.whl'):
            subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--force-reinstall', '--no-deps', whl], check=True)
    print('gsplat precompiled:', subprocess.run([sys.executable, '-c', 'from gsplat import csrc'], capture_output=True).returncode == 0)

    # F0.4: make gsplat's examples/datasets a regular package (beats HF `datasets`)
    open(f'{REPO}/examples/datasets/__init__.py', 'a').close()

    open(MARKER, 'w').write('ok')
    np_after = subprocess.run([sys.executable, '-c', 'import numpy; print(numpy.__version__)'],
                              capture_output=True, text=True).stdout.strip()
    if np_before != np_after:
        print(f'numpy {np_before} -> {np_after}: restarting runtime. Then "Run all" again.')
        time.sleep(2); os.kill(os.getpid(), 9)

REPO = f'{WORK}/gsplat'
check = subprocess.run([sys.executable, '-c',
    'import numpy, gsplat, fused_ssim, pycolmap, tyro; print("numpy", numpy.__version__, "| gsplat", gsplat.__version__)'],
    capture_output=True, text=True, cwd=f'{REPO}/examples')
print(check.stdout or check.stderr[-2000:])

## 3. Download one scene (~440 MB)

The scenes are public files in Wayve's Google Drive folder. We download one, keep a private copy on your Drive, and unzip it on the fast local disk.

If the download fails with a quota or permission error (Google Drive limits busy files), download `scene_078.zip` yourself from the folder linked at https://github.com/wayveai/wayve_scenes and put it in `MyDrive/3dgs-digital-twin/datasets/wayve/`.

In [ ]:
import urllib.request, urllib.parse, html, zipfile
FOLDER = '1XXlhGut7c5PVeBj4qwORlHoGdeSYxGgt'     # from the official wayve_scenes README
UA = {'User-Agent': 'Mozilla/5.0'}

def drive_file_id(name):
    page = urllib.request.urlopen(urllib.request.Request(
        f'https://drive.google.com/embeddedfolderview?id={FOLDER}', headers=UA)).read().decode('utf-8', 'replace')
    ids = {n: f for f, n in re.findall(r'id="entry-([\w-]+)".*?flip-entry-title">([^<]+)<', page, flags=re.S)}
    return ids[name]

def direct_url(fid):
    # Big files first return a "can't scan for viruses" page; its form holds the real download link
    u = f'https://drive.usercontent.google.com/download?id={fid}&export=download'
    body = urllib.request.urlopen(urllib.request.Request(u, headers={**UA, 'Range': 'bytes=0-0'})).read(200000)
    if body[:2] == b'PK':
        return u
    text = body.decode('utf-8', 'replace')
    form = re.search(r'<form[^>]+action="([^"]+)"', text)
    if not form:
        raise RuntimeError('Google Drive did not return a download link (quota or permissions). Download the zip manually, see above.')
    fields = dict(re.findall(r'<input type="hidden" name="([^"]+)" value="([^"]*)"', text))
    return html.unescape(form.group(1)) + '?' + urllib.parse.urlencode(fields)

ZIP_DRIVE = f'{CACHE}/{SCENE}.zip'
ZIP_LOCAL = f'{WORK}/{SCENE}.zip'
if not os.path.exists(ZIP_LOCAL):
    if os.path.exists(ZIP_DRIVE):
        print('copying cached zip from Drive'); shutil.copy(ZIP_DRIVE, ZIP_LOCAL)
    else:
        t = time.time(); done = 0
        with urllib.request.urlopen(urllib.request.Request(direct_url(drive_file_id(f'{SCENE}.zip')), headers=UA)) as r, \
             open(ZIP_LOCAL + '.part', 'wb') as f:
            while chunk := r.read(8 << 20):
                f.write(chunk); done += len(chunk)
                if done % (100 << 20) < (8 << 20):
                    print(f'  {done/1e6:.0f} MB')
        os.rename(ZIP_LOCAL + '.part', ZIP_LOCAL)
        print(f'downloaded {done/1e6:.0f} MB in {time.time()-t:.0f}s')
        shutil.copy(ZIP_LOCAL, ZIP_DRIVE)
assert zipfile.is_zipfile(ZIP_LOCAL), 'not a valid zip, delete it and retry'

RAW = f'{WORK}/wayve/{SCENE}'
if not os.path.exists(f'{RAW}/colmap_sparse/rig/images.bin'):
    with zipfile.ZipFile(ZIP_LOCAL) as z:
        z.extractall(f'{WORK}/wayve')
for sub in ['images', 'masks']:
    print(sub, {c: len(os.listdir(f'{RAW}/{sub}/{c}')) for c in sorted(os.listdir(f'{RAW}/{sub}'))})

## 4. Look at the data

In [ ]:
import numpy as np, cv2, imageio.v2 as imageio, matplotlib.pyplot as plt, csv, io
meta = list(csv.DictReader(io.StringIO(urllib.request.urlopen(
    'https://raw.githubusercontent.com/wayveai/wayve_scenes/main/data/scene_metadata.csv').read().decode())))
print({k: v for k, v in next(r for r in meta if r['scene_id'] == SCENE).items()})

CAMS = ['left-backward', 'left-forward', 'front-forward', 'right-forward', 'right-backward']
fig, ax = plt.subplots(2, 5, figsize=(22, 6))
for i, cam in enumerate(CAMS):
    files = sorted(os.listdir(f'{RAW}/images/{cam}'))
    for j, k in enumerate([0, len(files) - 1]):
        im = imageio.imread(f'{RAW}/images/{cam}/{files[k]}')
        m = imageio.imread(f'{RAW}/masks/{cam}/' + os.path.splitext(files[k])[0] + '.png') > 127
        im = im.copy(); im[~m] = (0.4 * im[~m] + [150, 0, 0]).astype(np.uint8)     # masked areas tinted red
        ax[j, i].imshow(im); ax[j, i].set_title(f"{cam} ({'start' if j == 0 else 'end'})", fontsize=9); ax[j, i].axis('off')
plt.tight_layout(); plt.show()

## 5. Undistort: fisheye → pinhole

For each camera we build an OpenCV fisheye undistortion map with the **same optical centre**, and a slightly shorter focal length for the wide side cameras (`ZOOM`) so more of the view survives. Then we downscale by `FACTOR` and write:
- `images/<camera>/<frame>.jpeg`: undistorted, 960×540
- `masks/<camera>/<frame>.png`: white = use this pixel. It combines Wayve's mask with the area the fisheye actually covers.
- `sparse/0/`: Wayve's poses and 3D points, plus a new `cameras.bin` with the matching **PINHOLE** cameras

This code was tested locally on sample frames of this scene before going into the notebook.

In [ ]:
import os, struct, shutil
import numpy as np, cv2
from concurrent.futures import ThreadPoolExecutor

N_PARAMS = {0: 3, 1: 4, 2: 4, 3: 5, 4: 8, 5: 8, 6: 12}
OPENCV_FISHEYE, PINHOLE = 5, 1
P2D = np.dtype([('x', '<f8'), ('y', '<f8'), ('id', '<i8')])

def read_cameras_bin(path):
    cams = {}
    with open(path, 'rb') as f:
        for _ in range(struct.unpack('<Q', f.read(8))[0]):
            cid, model = struct.unpack('<ii', f.read(8))
            w, h = struct.unpack('<QQ', f.read(16))
            cams[cid] = dict(model=model, w=w, h=h, params=np.array(struct.unpack(f'<{N_PARAMS[model]}d', f.read(8 * N_PARAMS[model]))))
    return cams

def write_cameras_bin(path, cams):
    with open(path, 'wb') as f:
        f.write(struct.pack('<Q', len(cams)))
        for cid, c in cams.items():
            f.write(struct.pack('<iiQQ', cid, c['model'], c['w'], c['h']))
            f.write(struct.pack(f'<{len(c["params"])}d', *c['params']))

def read_images_bin_full(path, with_points=False):
    imgs = {}
    with open(path, 'rb') as f:
        for _ in range(struct.unpack('<Q', f.read(8))[0]):
            iid = struct.unpack('<i', f.read(4))[0]
            q = np.array(struct.unpack('<4d', f.read(32))); t = np.array(struct.unpack('<3d', f.read(24)))
            cid = struct.unpack('<i', f.read(4))[0]
            name = b''
            while (c := f.read(1)) != b'\x00':
                name += c
            n2d = struct.unpack('<Q', f.read(8))[0]
            if with_points:
                pts = np.frombuffer(f.read(24 * n2d), dtype=P2D)
            else:
                f.seek(24 * n2d, 1); pts = None
            imgs[name.decode()] = dict(id=iid, q=q, t=t, cam=cid, pts=pts)
    return imgs

def read_points3D_xyz(path):
    ids, xyz = [], []
    with open(path, 'rb') as f:
        for _ in range(struct.unpack('<Q', f.read(8))[0]):
            ids.append(struct.unpack('<Q', f.read(8))[0])
            xyz.append(struct.unpack('<3d', f.read(24)))
            f.read(3 + 8)                                      # rgb + error
            f.seek(8 * struct.unpack('<Q', f.read(8))[0], 1)   # track
    return dict(zip(ids, range(len(ids)))), np.array(xyz)

def quat_to_R(q):
    w, x, y, z = q / np.linalg.norm(q)
    return np.array([[1-2*y*y-2*z*z, 2*x*y-2*w*z, 2*x*z+2*w*y],
                     [2*x*y+2*w*z, 1-2*x*x-2*z*z, 2*y*z-2*w*x],
                     [2*x*z-2*w*y, 2*y*z+2*w*x, 1-2*x*x-2*y*y]])

def fisheye_K_D(cam):
    fx, fy, cx, cy, k1, k2, k3, k4 = cam['params']
    return np.array([[fx, 0, cx], [0, fy, cy], [0, 0, 1.0]]), np.array([k1, k2, k3, k4])

def pinhole_camera(cam, factor, zoom=1.0):
    # Same optical centre as the fisheye. zoom < 1 shortens the focal length to keep more of the wide view.
    # Downscaling by `factor` follows the pixel-centre convention used by cv2.resize.
    fx, fy, cx, cy = cam['params'][:4]
    w, h = round(cam['w'] / factor), round(cam['h'] / factor)
    sx, sy = w / cam['w'], h / cam['h']
    return dict(model=PINHOLE, w=w, h=h, params=np.array([fx * zoom * sx, fy * zoom * sy, (cx + 0.5) * sx - 0.5, (cy + 0.5) * sy - 0.5]))

def undistort_maps(cam, zoom=1.0):
    K, D = fisheye_K_D(cam)
    P = K.copy(); P[0, 0] *= zoom; P[1, 1] *= zoom
    mapx, mapy = cv2.fisheye.initUndistortRectifyMap(K, D, np.eye(3), P, (cam['w'], cam['h']), cv2.CV_32FC1)
    valid = (mapx >= 0) & (mapy >= 0) & (mapx <= cam['w'] - 1) & (mapy <= cam['h'] - 1)
    return mapx, mapy, valid

def zoom_for(cam, zoom):
    # Zoom out only the wide side cameras (focal < 0.6 x width). The narrow front camera would get black borders.
    return zoom if cam['params'][0] < 0.6 * cam['w'] else 1.0

def prepare_scene(raw, out, factor, zoom=1.0, threads=4):
    # raw: unzipped Wayve scene (images/, masks/, colmap_sparse/rig/). out: training-ready folder.
    cams = read_cameras_bin(f'{raw}/colmap_sparse/rig/cameras.bin')
    assert all(c['model'] == OPENCV_FISHEYE for c in cams.values()), 'expected OPENCV_FISHEYE cameras'
    imgs = read_images_bin_full(f'{raw}/colmap_sparse/rig/images.bin')
    maps = {cid: undistort_maps(c, zoom_for(c, zoom)) for cid, c in cams.items()}
    new = {cid: pinhole_camera(c, factor, zoom_for(c, zoom)) for cid, c in cams.items()}
    os.makedirs(f'{out}/sparse/0', exist_ok=True)
    write_cameras_bin(f'{out}/sparse/0/cameras.bin', new)
    for fn in ['images.bin', 'points3D.bin']:
        shutil.copy(f'{raw}/colmap_sparse/rig/{fn}', f'{out}/sparse/0/{fn}')

    def one(name):
        src = f'{raw}/images/{name}'
        if not os.path.exists(src):
            return None
        cid = imgs[name]['cam']; mapx, mapy, valid = maps[cid]; size = (new[cid]['w'], new[cid]['h'])
        im = cv2.remap(cv2.imread(src), mapx, mapy, cv2.INTER_LINEAR, borderMode=cv2.BORDER_CONSTANT)
        dst = f'{out}/images/{name}'
        os.makedirs(os.path.dirname(dst), exist_ok=True)
        cv2.imwrite(dst, cv2.resize(im, size, interpolation=cv2.INTER_AREA), [cv2.IMWRITE_JPEG_QUALITY, 95])
        # keep-mask: Wayve mask (white = usable) AND inside the fisheye image; conservative after downscaling
        mpath = f'{raw}/masks/' + os.path.splitext(name)[0] + '.png'
        keep = cv2.imread(mpath, cv2.IMREAD_GRAYSCALE) if os.path.exists(mpath) else np.full((cams[cid]['h'], cams[cid]['w']), 255, np.uint8)
        keep = cv2.remap(keep, mapx, mapy, cv2.INTER_NEAREST, borderMode=cv2.BORDER_CONSTANT) * valid
        keep = cv2.resize((keep > 127).astype(np.float32), size, interpolation=cv2.INTER_AREA) > 0.99
        mdst = f'{out}/masks/' + os.path.splitext(name)[0] + '.png'
        os.makedirs(os.path.dirname(mdst), exist_ok=True)
        cv2.imwrite(mdst, keep.astype(np.uint8) * 255)
        return float(keep.mean())

    with ThreadPoolExecutor(threads) as ex:
        kept = [k for k in ex.map(one, sorted(imgs)) if k is not None]
    return cams, new, imgs, kept

def reprojection_check(raw, imgs_sel, cams, pid_index, xyz, out=None, new=None, max_pts=2000):
    # Projects COLMAP's 3D points into each image and compares with the stored 2D observations.
    # Fisheye: original model on original pixels. Pinhole (if `out`/`new` given): the same check after our undistortion.
    res = {}
    for name in imgs_sel:
        im = imgs_sel[name]; cam = cams[im['cam']]; K, D = fisheye_K_D(cam)
        obs = im['pts'][im['pts']['id'] >= 0][:max_pts]
        X = xyz[[pid_index[i] for i in obs['id']]]
        rvec, _ = cv2.Rodrigues(quat_to_R(im['q']))
        uv = np.stack([obs['x'], obs['y']], axis=1)
        proj, _ = cv2.fisheye.projectPoints(X.reshape(-1, 1, 3), rvec, im['t'], K, D)
        r = {'fisheye_px': float(np.median(np.linalg.norm(proj.reshape(-1, 2) - uv, axis=1)))}
        if new is not None:
            fx, fy, cx, cy = new[im['cam']]['params']
            Kp = np.array([[fx, 0, cx], [0, fy, cy], [0, 0, 1.0]])
            # observations moved into the undistorted, downscaled image
            und = cv2.fisheye.undistortPoints(uv.reshape(-1, 1, 2), K, D, P=Kp).reshape(-1, 2)
            pin, _ = cv2.projectPoints(X, rvec, im['t'], Kp, None)
            inside = (und[:, 0] >= 0) & (und[:, 0] < new[im['cam']]['w']) & (und[:, 1] >= 0) & (und[:, 1] < new[im['cam']]['h'])
            r['pinhole_px'] = float(np.median(np.linalg.norm(pin.reshape(-1, 2)[inside] - und[inside], axis=1)))
            r['obs_inside_crop_pct'] = round(100 * float(inside.mean()), 1)
        res[name] = r
    return res


SCENE_DIR = f'{WORK}/data/wayve_{SCENE}'
DONE = f'{SCENE_DIR}/.prepared'
if not os.path.exists(DONE):
    shutil.rmtree(SCENE_DIR, ignore_errors=True)
    t = time.time()
    cams, new, imgs, kept = prepare_scene(RAW, SCENE_DIR, FACTOR, ZOOM)
    open(DONE, 'w').write('ok')
    print(f'prepared {len(kept)} images in {time.time()-t:.0f}s | usable pixels per image: mean {np.mean(kept):.3f}, min {np.min(kept):.3f}')
cams = read_cameras_bin(f'{RAW}/colmap_sparse/rig/cameras.bin')
new = read_cameras_bin(f'{SCENE_DIR}/sparse/0/cameras.bin')
for cid in sorted(cams):
    print(f"camera {cid}: fisheye f={cams[cid]['params'][0]:.0f} -> pinhole {new[cid]['w']}x{new[cid]['h']} f={new[cid]['params'][0]:.1f}")

## 6. Pose QA: reproject Wayve's 3D points

We can't compare against a reference here, but the dataset stores, for each image, which 3D point was seen at which pixel. Projecting the 3D points back with the camera model should land on those pixels:
- **fisheye:** the original images with the original camera model, which checks Wayve's poses and our reading of the camera model
- **pinhole:** the same points after our undistortion, which checks our preparation step

Local test on this scene: fisheye 0.5-0.75 px, pinhole 0.3-0.4 px (half resolution). The simplified model gsplat uses for fisheye: ~83 px.

In [ ]:
full = read_images_bin_full(f'{RAW}/colmap_sparse/rig/images.bin', with_points=True)
pid, xyz = read_points3D_xyz(f'{RAW}/colmap_sparse/rig/points3D.bin')
sample = {n: full[n] for n in sorted(full)[::20]}
res = reprojection_check(RAW, sample, cams, pid, xyz, SCENE_DIR, new)
by_cam = {}
for n, r in res.items():
    by_cam.setdefault(n.split('/')[0], []).append(r)
print(f"{'camera':>15} | {'fisheye px':>10} | {'pinhole px':>10} | points in frame")
for cam, rs in sorted(by_cam.items()):
    print(f"{cam:>15} | {np.median([r['fisheye_px'] for r in rs]):10.2f} | {np.median([r['pinhole_px'] for r in rs]):10.2f} | "
          f"{np.median([r['obs_inside_crop_pct'] for r in rs]):.0f}%")
fe = np.median([r['fisheye_px'] for r in res.values()]); pe = np.median([r['pinhole_px'] for r in res.values()])
print(f"\nmedian fisheye {fe:.2f} px {'✅' if fe < 1.5 else '❌'} | median pinhole {pe:.2f} px {'✅' if pe < 1.0 else '❌'}")

# Camera paths, top-down
centers = {}
for n, im in full.items():
    R = quat_to_R(im['q']); centers.setdefault(n.split('/')[0], []).append(-R.T @ im['t'])
allc = np.concatenate([np.array(v) for v in centers.values()])
_, _, V = np.linalg.svd(allc - allc.mean(0), full_matrices=False)
plt.figure(figsize=(12, 4))
for cam, cs in sorted(centers.items()):
    P = (np.array(cs) - allc.mean(0)) @ V[:2].T
    plt.plot(P[:, 0], P[:, 1], '.', ms=2, label=cam)
plt.axis('equal'); plt.legend(markerscale=6); plt.title('Camera positions (top-down)'); plt.show()
json.dump({'reprojection_by_camera': {c: {k: float(np.median([r[k] for r in rs])) for k in rs[0]} for c, rs in by_cam.items()}},
          open(f'{OUT}/pose_qa.json', 'w'), indent=1)

In [ ]:
# Before / after undistortion for one moment in time
stamp = sorted(os.listdir(f'{RAW}/images/front-forward'))[100]
fig, ax = plt.subplots(3, 5, figsize=(22, 7))
for i, cam in enumerate(CAMS):
    ax[0, i].imshow(imageio.imread(f'{RAW}/images/{cam}/{stamp}')); ax[0, i].set_title(f'{cam}: original', fontsize=9)
    ax[1, i].imshow(imageio.imread(f'{SCENE_DIR}/images/{cam}/{stamp}')); ax[1, i].set_title('undistorted', fontsize=9)
    ax[2, i].imshow(imageio.imread(f'{SCENE_DIR}/masks/{cam}/' + os.path.splitext(stamp)[0] + '.png'), cmap='gray'); ax[2, i].set_title('mask (white = used)', fontsize=9)
for a in ax.flat: a.axis('off')
plt.tight_layout(); plt.show()

## 7. Teach gsplat's data loader two things

A small, idempotent patch to `gsplat/examples/datasets/colmap.py`, switched on only by environment variables (the garden and phone notebooks are unaffected):
1. **`GS_TEST_PREFIX`:** hold out a whole camera for testing (`front-forward/`) instead of every 8th image.
2. **`GS_MASK_DIR`:** load our masks. Masked pixels are set to black in the photo, and gsplat already blacks out the render where the mask is false, so they add nothing to the loss or the scores.

In [ ]:
LOADER = f'{REPO}/examples/datasets/colmap.py'
src = open(LOADER).read()
if 'GS_TEST_PREFIX' not in src:
    split_old = '''        indices = np.arange(len(self.parser.image_names))
        if split == "train":
            self.indices = indices[indices % self.parser.test_every != 0]
        else:
            self.indices = indices[indices % self.parser.test_every == 0]'''
    split_new = '''        indices = np.arange(len(self.parser.image_names))
        test_prefix = os.environ.get("GS_TEST_PREFIX")  # hold out a whole camera, e.g. "front-forward/"
        if test_prefix:
            is_test = np.array([n.startswith(test_prefix) for n in self.parser.image_names])
            stride = int(os.environ.get("GS_TEST_STRIDE", "1"))
            self.indices = indices[~is_test] if split == "train" else indices[is_test][::stride]
        elif split == "train":
            self.indices = indices[indices % self.parser.test_every != 0]
        else:
            self.indices = indices[indices % self.parser.test_every == 0]'''
    mask_anchor = '''        if self.patch_size is not None:
            # Random crop.'''
    mask_new = '''        mask_dir = os.environ.get("GS_MASK_DIR")  # white = use this pixel
        if mask_dir:
            stem = os.path.splitext(self.parser.image_names[index])[0]
            keep = imageio.imread(os.path.join(mask_dir, stem + ".png"))
            keep = keep[..., 0] if keep.ndim == 3 else keep
            if keep.shape[:2] != image.shape[:2]:
                keep = cv2.resize(keep, (image.shape[1], image.shape[0]), interpolation=cv2.INTER_NEAREST)
            keep = keep > 127
            image = image * keep[..., None].astype(image.dtype)
            mask = keep if mask is None else (mask & keep)

''' + mask_anchor
    assert src.count(split_old) == 1 and src.count(mask_anchor) == 1, 'gsplat loader changed, patch needs updating'
    src = src.replace(split_old, split_new).replace(mask_anchor, mask_new)
    open(LOADER, 'w').write(src)
    print('loader patched')

os.environ['GS_TEST_PREFIX'] = 'front-forward/'
os.environ['GS_TEST_STRIDE'] = str(TEST_STRIDE)
os.environ['GS_MASK_DIR'] = f'{SCENE_DIR}/masks'
check = subprocess.run([sys.executable, '-c', f'''
from datasets.colmap import Parser, Dataset
p = Parser("{SCENE_DIR}", factor=1, normalize=True, test_every=8)
tr, te = Dataset(p, "train"), Dataset(p, "val")
tr_names = [p.image_names[i] for i in tr.indices]; te_names = [p.image_names[i] for i in te.indices]
assert not any(n.startswith("front-forward/") for n in tr_names), "front camera leaked into training"
assert all(n.startswith("front-forward/") for n in te_names)
d = tr[0]
print("train", len(tr), "| test", len(te), "| image", tuple(d["image"].shape), "| mask", tuple(d["mask"].shape),
      "| used pixels %.3f" % d["mask"].float().mean().item())
'''], capture_output=True, text=True, cwd=f'{REPO}/examples')
print(check.stdout or check.stderr[-3000:])

## 8. Train (15k steps, half of gsplat's standard schedule)

`--steps_scaler 0.5` shortens every phase of training together (adding Gaussians, opacity resets, colour-detail warm-up), so this is a proper short run rather than a long run cut off halfway.

In [ ]:
TRAIN_EXTRA = ['--steps_scaler', str(SCALE)]
help_txt = subprocess.run([sys.executable, 'simple_trainer.py', 'default', '--help'], capture_output=True,
                          text=True, cwd=f'{REPO}/examples').stdout
LPIPS = ['--lpips_net', 'vgg'] if re.search(r'lpips.net', help_txt) else []
print('LPIPS flag:', LPIPS or 'not available (AlexNet default)')

def check_resolution(rd, data_dir):
    # eval renders are GT | render side by side, so each half must match the input image width
    expected = imageio.imread(sorted(f for f in glob.glob(f'{data_dir}/images/**/*', recursive=True) if os.path.isfile(f))[0]).shape[1]
    actual = imageio.imread(sorted(glob.glob(f'{rd}/renders/val_step*_*.png'))[0]).shape[1] / 2
    print(f'   trained at width {actual:.0f}px (input images: {expected}px)')
    if abs(actual - expected) / expected > 0.1:
        raise RuntimeError(f'Trained at {actual:.0f}px instead of {expected}px: wrong data_factor? See F1.2')

def train(run, data_dir, factor):
    rd = f'{PROJECT}/results/{run}'
    stats = sorted(glob.glob(f'{rd}/stats/val_step*.json'))
    if stats:
        print(f'✔ {run}: already trained, loading stats'); check_resolution(rd, data_dir)
        s = json.load(open(stats[-1])); s['train_min'] = None; return s
    cmd = [sys.executable, 'simple_trainer.py', 'default', '--data_dir', data_dir, '--data_factor', str(factor),
           '--result_dir', rd, '--max_steps', str(STEPS), '--eval_steps', str(STEPS), '--save_steps', str(STEPS),
           '--disable_viewer', *LPIPS, *globals().get('TRAIN_EXTRA', [])]
    os.makedirs(rd, exist_ok=True)
    t0 = last = time.time(); buf = ''; tail = []
    p = subprocess.Popen(cmd, cwd=f'{REPO}/examples', stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    with open(f'{rd}/train_log.txt', 'w') as lf:
        while (chunk := p.stdout.read(512)):
            lf.write(chunk); buf += chunk
            parts = re.split(r'[\r\n]', buf); buf = parts[-1]
            for ln in filter(str.strip, parts[:-1]):
                tail = (tail + [ln])[-40:]
                if time.time() - last > 60:
                    print(f'[{(time.time()-t0)/60:4.1f} min] {ln.strip()[:140]}'); last = time.time()
    if p.wait() != 0:
        print('\n'.join(tail)); raise RuntimeError(f'training failed, log: {rd}/train_log.txt')
    s = json.load(open(sorted(glob.glob(f'{rd}/stats/val_step*.json'))[-1]))
    s['train_min'] = round((time.time() - t0) / 60, 1)
    check_resolution(rd, data_dir)
    print(f"✔ {run}: PSNR {s['psnr']:.2f} | SSIM {s['ssim']:.3f} | LPIPS {s['lpips']:.3f} | "
          f"{s.get('num_GS', 0):,} GS | {s['train_min']} min")
    return s

RUN = f'wayve_{SCENE}_{int(STEPS * SCALE) // 1000}k'
RESULT_DIR = f'{PROJECT}/results/{RUN}'
m = train(RUN, SCENE_DIR, 1)
print({k: (round(v, 4) if isinstance(v, float) else v) for k, v in m.items()})

## 9. Results on the held-out front camera

⚠️ **This is a harder test than garden or the phone video.** The front camera was never trained on, it has a narrower lens than the side cameras, and it looks straight down the road, where the side cameras only saw things at an angle. Expect clearly lower scores, and don't compare them with garden's.

In [ ]:
for f in sorted(glob.glob(f'{RESULT_DIR}/renders/val_step*_*.png'))[::25][:4]:
    plt.figure(figsize=(18, 5)); plt.imshow(imageio.imread(f)); plt.axis('off'); plt.title(os.path.basename(f) + '  (real | render)'); plt.show()
videos = glob.glob(f'{RESULT_DIR}/videos/*.mp4')
print('fly-through video:', videos or 'none')

## 10. Export (private) and log the run

In [ ]:
import torch, csv, datetime
from plyfile import PlyData, PlyElement

def ckpt_to_ply(ckpt_path, ply_path):
    s = torch.load(ckpt_path, map_location='cpu', weights_only=False)['splats']
    means = s['means'].numpy()
    sh0 = s['sh0'].transpose(1, 2).flatten(1).numpy()            # [N,1,3] -> [N,3]
    shN = s['shN'].transpose(1, 2).flatten(1).numpy()            # [N,K,3] -> [N,3K] (channel-major, 3DGS convention)
    opac = s['opacities'].numpy()[:, None]                       # logit space
    scales = s['scales'].numpy()                                 # log space
    quats = s['quats'].numpy()                                   # wxyz
    normals = np.zeros_like(means)
    attrs = ['x','y','z','nx','ny','nz'] + [f'f_dc_{i}' for i in range(sh0.shape[1])] \
          + [f'f_rest_{i}' for i in range(shN.shape[1])] + ['opacity'] \
          + [f'scale_{i}' for i in range(3)] + [f'rot_{i}' for i in range(4)]
    data = np.concatenate([means, normals, sh0, shN, opac, scales, quats], axis=1).astype(np.float32)
    el = np.empty(data.shape[0], dtype=[(a, 'f4') for a in attrs])
    el[:] = list(map(tuple, data))
    PlyData([PlyElement.describe(el, 'vertex')]).write(ply_path)
    return data.shape[0]

ckpts = sorted(glob.glob(f'{RESULT_DIR}/ckpts/*.pt'))
assert ckpts, 'No checkpoint found'
PLY_PATH = f'{PROJECT}/exports/{RUN}.ply'
n = ckpt_to_ply(ckpts[-1], PLY_PATH)
print(f'{n:,} Gaussians → {PLY_PATH} ({os.path.getsize(PLY_PATH)/1e6:.0f} MB)')
print('⚖️ Keep this file private (licence clause 5a). Share videos or screenshots, with the attribution text.')

CSV = f'{PROJECT}/results/metrics.csv'
header = next(csv.reader(open(CSV))) if os.path.exists(CSV) else \
    ['date', 'run', 'scene', 'res', 'strategy', 'steps', 'num_gaussians', 'psnr', 'ssim', 'lpips', 'gpu', 'mem_gb', 'ply_mb']
rows = [r for r in csv.DictReader(open(CSV)) if r['run'] != RUN] if os.path.exists(CSV) else []
rows.append({'date': str(datetime.date.today()), 'run': RUN, 'scene': f'wayve {SCENE} (front cam held out)',
             'res': '960x540', 'strategy': 'default', 'steps': int(STEPS * SCALE), 'num_gaussians': n,
             'psnr': round(m['psnr'], 3), 'ssim': round(m['ssim'], 4),
             'lpips': f"{round(m['lpips'], 4)} ({'vgg' if LPIPS else 'alex'})", 'gpu': torch.cuda.get_device_name(0),
             'mem_gb': m.get('mem', ''), 'ply_mb': round(os.path.getsize(PLY_PATH) / 1e6, 1)})
with open(CSV, 'w', newline='') as f:
    w_ = csv.DictWriter(f, fieldnames=header, extrasaction='ignore'); w_.writeheader(); w_.writerows(rows)
print('logged to', CSV)

from google.colab import files
files.download(PLY_PATH)

## ✅ Checklist
- [ ] Reprojection: fisheye median < 1.5 px, pinhole median < 1 px
- [ ] Loader check: train 800, test 100, and the front camera never in training
- [ ] Training finished; the trained width printed as 960
- [ ] The `.ply` opens in SuperSplat and looks like the street from **where the car drove**
- [ ] Anything that broke → `docs/failures.md`

**Paste back into the chat:** the outputs of sections 6, 7 and 8, plus a SuperSplat screenshot from the road.

*This work was made using the WayveScenes101 dataset, provided by Wayve Technologies Ltd under licence terms available at https://wayve.ai/science/wayvescenes101/*